In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType, IntegerType
from pyspark.sql.functions import col, regexp_replace, nullif, lit, to_date

# --- Định nghĩa các hàm Clean dữ liệu ---
def clean_decimal(c, p=38, s=8):
    # Loại bỏ ký tự không phải số để cast về Decimal
    cleaned = regexp_replace(col(c), "[^0-9.]", "") 
    return nullif(cleaned, lit("")).cast(DecimalType(p, s)).alias(c)

def clean_int(c):
    cleaned = regexp_replace(col(c), "[^0-9]", "")
    return when(
        (col(c).isNull()) | (length(cleaned) == 0),
        None
    ).otherwise(cleaned.cast(IntegerType())).alias(c)

def clean_date(c):
    # Hỗ trợ định dạng YYYY-DD-MM hoặc các định dạng ngày phổ biến khác
    return coalesce(
        to_date(col(c), 'yyyy-dd-MM'),
        to_date(col(c), 'yyyy-MM-dd'),
        to_date(col(c), 'M/d/yyyy'),
        to_date(col(c), 'MM/dd/yyyy')
    ).alias(c)

# --- Quá trình Sync Data ---

# 1. Đọc dữ liệu từ bảng nguồn app_branch
df_src = spark.read.table("mb_poc.data_raw.app_branch")

# 2. Transform và Mapping dữ liệu
df_tgt = (
    df_src.select(
        clean_date("CDR_DT"),
        col("ITM").alias("ITM"),
        clean_int("KPI"),
        col("RM_CODE").alias("RM_CODE"),
        col("RM_NM").alias("RM_NM"),
        col("SUB_BRANCH_CODE").alias("SUB_BRANCH_CODE"),
        col("SUB_BRANCH_NAME").alias("SUB_BRANCH_NAME"),
        col("BRANCH_CODE").alias("BRANCH_CODE"),
        col("BRANCH_NAME").alias("BRANCH_NAME"),
        col("SUB_RGON").alias("SUB_RGON"),
        col("PRN_OU_TP").alias("PRN_OU_TP"),
        col("CNL").alias("CNL"),
        clean_int("NBR_APP_NEW_D"),
        clean_int("NBR_APP_NEW_MTD"),
        clean_int("NBR_APP_NEW_YTD"),
        clean_int("APP_KPI_MTD"),
        clean_int("APP_KPI_YTD"),
        clean_int("NBR_NFC_SCS")
    )
)
#df_tgt.display()
# 3. Ghi dữ liệu vào bảng đích Gold
df_tgt.write.mode("append").insertInto("mb_poc.gold.rpt_nhs_d01_kpi_branch")